<a href="https://colab.research.google.com/github/jeromelin0933/NongHui_RAG_Project/blob/main/%E8%BE%B2%E6%9C%83%E8%A6%8F%E7%AB%A0%E5%8A%A9%E6%89%8B_Phase1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain-text-splitters langchain-core

In [ ]:
import antigravity # 專案起飛！

# 由於 Colab 每次重啟都會清空環境，我們把安裝指令寫在第一個 Cell
print("🚀 正在啟動引擎並安裝核心套件 (這可能需要 1-2 分鐘)...")

# 安裝 Docling (PDF 解析神器)
!pip install -q docling

# 安裝 LangChain 相關基礎套件 (Phase 2 切片時會用到)
!pip install -q langchain-text-splitters
!pip install -q langchain-community

print("✅ 套件安裝完畢！")

In [ ]:
# 測試 Docling 是否能順利載入
try:
    from docling.document_converter import DocumentConverter
    print("🎉 驗收成功：Docling 模組載入正常！你的雲端解析環境已經準備就緒。")
except ImportError as e:
    print(f"❌ 載入失敗，請檢查安裝步驟。錯誤訊息：{e}")

In [ ]:
import time
from docling.document_converter import DocumentConverter

# 設定你的假資料檔案名稱
file_path = "Mock_Rule_A.pdf"

print(f"🚀 啟動 Docling 視覺模型，開始深度解析 {file_path} ... (請稍候)")
start_time = time.time()

# 1. 初始化文件轉換器
converter = DocumentConverter()

# 2. 執行轉換 (這步驟會吃 GPU 算力來辨識表格和排版)
result = converter.convert(file_path)

# 3. 將解析完的結構化物件，降維匯出為我們 RAG 最愛的 Markdown 格式
markdown_text = result.document.export_to_markdown()

end_time = time.time()
print(f"✅ 解析完成！總耗時: {end_time - start_time:.2f} 秒")

In [ ]:
# 存檔設定
output_filename = "Mock_Rule_A_cleaned.md"

# 寫入本機端
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(markdown_text)

# 印出前半段內容來做人工驗收
print("👀 預覽轉換後的 Markdown 內容：\n")
print("=" * 60)
print(markdown_text[:1500]) # 印出前 1500 字確認結構
print("=" * 60)

# 觸發瀏覽器下載，把這份珍貴的乾淨資料存回你的筆電
from google.colab import files
files.download(output_filename)

In [ ]:
import re
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 讀取 Phase 1 產出的乾淨 Markdown 檔案
file_path = "Mock_Rule_A_cleaned.md"
with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("✂️ 開始進行階層式切片與麵包屑標記...")

documents = []
current_chapter = "未分類規章" # 預設章節
current_article = "未分類條文" # 預設條文
current_chunk_text = []

# 2. 定義儲存 Chunk 的輔助函式
def save_chunk():
    if current_chunk_text:
        text = "".join(current_chunk_text).strip()
        if text:
            # 組合麵包屑 (例如：第一章 總則 > 第一條 (目的))
            breadcrumb = f"{current_chapter} > {current_article}".strip(" > ")
            # 建立 LangChain 的 Document 物件，並將麵包屑塞進 metadata
            documents.append(Document(page_content=text, metadata={"breadcrumb": breadcrumb}))
        current_chunk_text.clear()

# 3. 逐行掃描，利用 Regex 捕捉章節與條文
for line in lines:
    # 抓取章節 (例如：## 第⼀章 總則) - 處理可能的空白與全半角差異
    if re.match(r'^##\s*第[一二三四五六七八九十百]+章', line.replace('⼀', '一').replace('⼆', '二')):
        save_chunk() # 遇到新章節，先把前面的內容存起來
        current_chapter = line.replace('##', '').strip()
        current_article = "" # 換章節時，條文清空

    # 抓取條文 (例如：## 第三條 ( 開⼾情境... ))
    elif re.match(r'^##\s*第[一二三四五六七八九十百]+條', line.replace('⼀', '一').replace('⼆', '二').replace('三', '三')):
        save_chunk() # 遇到新條文，先把前面的內容存起來
        current_article = line.replace('##', '').strip()

    else:
        # 如果不是標題，就是內文，加入當前的 chunk 中
        current_chunk_text.append(line)

save_chunk() # 存入最後一塊

# 4. 二次切片防護機制 (如果某個條文的表格或文字太長，再用字數切一刀)
# 設定每塊最大 800 字，保留 100 字的重疊(overlap)避免上下文斷裂
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)
final_chunks = text_splitter.split_documents(documents)

print(f"✅ 切片完成！共切出 {len(final_chunks)} 個知識區塊 (Chunks)。")

In [ ]:
!pip install -q langchain-community langchain

In [ ]:
# 1. 改安裝 faiss-cpu (超級輕量，不會有依賴衝突)
!pip install -q sentence-transformers faiss-cpu langchain-community

from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import FAISS # 替換為 FAISS

print("🧠 正在載入 BAAI/bge-m3 模型 (第一次下載約需 2-3 分鐘，請稍候)...")
# 設定 Embedding 模型
model_name = "BAAI/bge-m3"
model_kwargs = {'device': 'cuda'} # 呼叫 Colab 的 T4 GPU 來加速
encode_kwargs = {'normalize_embeddings': True}

embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

print("🗄️ 正在建立 FAISS 向量資料庫...")
# 2. 將切好的 final_chunks 轉換為向量並存入 FAISS
vectorstore = FAISS.from_documents(
    documents=final_chunks,
    embedding=embeddings
)

# 儲存到本地端備用
vectorstore.save_local("./faiss_index")
print("✅ FAISS 資料庫建置完成！(再也不用看那堆紅字了 🎉)")

print("="*50)
print("🎯 模擬農會櫃員提問測試 (AC Check)")
print("="*50)

# 3. 測試檢索能力：模擬一段模糊的白話文提問
query = "如果是阿公帶七歲以下的孫子來開戶，要準備什麼？"
print(f"🙋 櫃員提問: {query}\n")

# 尋找最接近的 2 塊法規 (Top-K=2)
retrieved_docs = vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(retrieved_docs):
    print(f"🔍 【命中排名 {i+1}】")
    print(f"📍 出處: {doc.metadata.get('breadcrumb', '未分類')}")
    print(f"📄 內文擷取: {doc.page_content[:150]}...")
    print("-" * 30)

In [ ]:
!pip install -q langchain-google-genai

In [ ]:
import os
import getpass
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# 1. 安全輸入 Google API Key
print("🔑 請輸入你的 Google Gemini API Key：")
os.environ["GOOGLE_API_KEY"] = getpass.getpass()

# 2. 初始化 Gemini 2.5 Flash 模型 (替換為最新版模型)
print("🧠 正在喚醒 Gemini 2.5 Flash...")
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", # <--- 只有這裡改了！把 1.5 改成 2.5
    temperature=0.1,
)

# 3. 撰寫「情境分流」專用的 System Prompt
prompt_template = ChatPromptTemplate.from_template("""
你是一位專業且嚴謹的農會信用部櫃檯助手。
請「僅根據」以下提供的【法規資料】來回答櫃員的提問。

【回答核心要求】（請嚴格遵守）：
1. 依據情境分流：針對複雜問題，請務必拆分不同情境（例如：狀況一、狀況二）條列式說明，不要把資訊糊在一起。
2. 絕對不捏造：如果法規資料中沒有明確提到答案，請直接回答「根據現有內部規章，未找到相關資訊」，嚴禁自行補充外部知識。
3. 標註來源：請在回答的最後，獨立換行標註「📑 參考法規出處：」，並列出你參考的法規麵包屑路徑。

【法規資料】：
{context}

【櫃員提問】：
{question}
""")

# 4. 定義組裝上下文的輔助函式
def format_docs(docs):
    formatted_texts = []
    source_breadcrumbs = set() # 用 set 避免重複出處

    for doc in docs:
        formatted_texts.append(doc.page_content)
        source_breadcrumbs.add(doc.metadata.get('breadcrumb', '未知來源'))

    context_str = "\n\n".join(formatted_texts)
    sources_str = " / ".join(source_breadcrumbs)
    return context_str, sources_str

print("="*50)
print("🤖 農會 AI 規章助手 (正式組裝版) 測試啟動")
print("="*50)

# 5. 執行完整的 RAG 流程 (檢索 + 生成)
query = "如果是阿公帶七歲以下的孫子來開戶，要準備什麼？阿公可以代簽嗎？"
print(f"🙋 櫃員提問：{query}\n")
print("⏳ AI 思考與彙整中...\n")

# 步驟 A: 檢索
retrieved_docs = vectorstore.similarity_search(query, k=3)
context_text, sources = format_docs(retrieved_docs)

import time

# 步驟 B & C: 組裝 Prompt、呼叫 LLM 並加入自動重試機制
final_prompt = prompt_template.format_messages(
    context=context_text,
    question=query
)

max_retries = 3  # 最多重試 3 次
retry_delay = 5  # 每次重試間隔 5 秒

for attempt in range(max_retries):
    try:
        response = llm.invoke(final_prompt)

        # 如果成功拿到答案，就印出來並跳出迴圈
        print("✅ 成功取得 AI 回答：\n")
        print(response.content)
        print("\n" + "-"*40)
        print(f"📑 參考法規出處：{sources}")
        break

    except Exception as e:
        # 捕捉到伺服器忙碌等錯誤
        print(f"⚠️ 伺服器目前忙碌中 (嘗試 {attempt + 1}/{max_retries})... 等待 {retry_delay} 秒後自動重試...")
        time.sleep(retry_delay)
else:
    # 如果 3 次都失敗，給出友善的錯誤提示
    print("\n❌ Google API 伺服器持續滿載，請喝杯水，稍後再重新執行一次。")

In [ ]:
# 將 FAISS 資料庫打包成 zip
!zip -r faiss_index.zip ./faiss_index

# 下載到你的筆電 (請務必確認瀏覽器有成功下載！)
from google.colab import files
files.download("faiss_index.zip")

In [ ]:
%%writefile app.py
import streamlit as st
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import FAISS

# 1. 頁面設定
st.set_page_config(page_title="農會規章 AI 助手", page_icon="🌾", layout="centered")
st.title("🌾 農會信用部 AI 規章助手")
st.markdown("**專屬第一線櫃員的法規檢索系統 (Demo 版)**")

# 2. 側邊欄：設定 API Key
with st.sidebar:
    st.header("⚙️ 系統設定")
    api_key = st.text_input("輸入 Google Gemini API Key:", type="password")
    st.markdown("---")
    st.info("💡 系統狀態：FAISS 向量庫已就緒\n💡 嵌入模型：bge-m3 (地端)")

# 初始化暫存對話紀錄
if "messages" not in st.session_state:
    st.session_state.messages = []

# 顯示歷史對話
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        # 如果是 AI 回答且有附帶來源，用摺疊面板顯示原文
        if msg["role"] == "assistant" and "sources" in msg:
            with st.expander("📄 查看檢索到的法規原文"):
                st.markdown(msg["sources"])

# 3. 處理使用者輸入
if prompt := st.chat_input("請輸入法規問題 (例如: 未成年開戶要準備什麼？)"):
    if not api_key:
        st.warning("請先在左側輸入 Gemini API Key！")
        st.stop()

    # 顯示使用者問題
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # 4. 啟動 RAG 流程
    with st.chat_message("assistant"):
        with st.spinner("檢索內部規章中..."):
            os.environ["GOOGLE_API_KEY"] = api_key

            # 載入地端模型與資料庫
            embeddings = HuggingFaceBgeEmbeddings(model_name="BAAI/bge-m3", model_kwargs={'device': 'cpu'})
            vectorstore = FAISS.load_local("./faiss_index", embeddings, allow_dangerous_deserialization=True)
            llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)

            # 檢索
            docs = vectorstore.similarity_search(prompt, k=3)
            context_text = "\n\n".join([d.page_content for d in docs])
            source_breadcrumbs = set([d.metadata.get('breadcrumb', '未知來源') for d in docs])

            # 整理來源顯示用的 Markdown
            source_display = ""
            for i, doc in enumerate(docs):
                source_display += f"**📍 {doc.metadata.get('breadcrumb', '未知來源')}**\n\n> {doc.page_content}\n\n---\n"

            # Prompt 模板
            prompt_template = ChatPromptTemplate.from_template("""
            你是一位專業且嚴謹的農會信用部櫃檯助手。
            請「僅根據」以下提供的【法規資料】來回答提問。如果沒有資訊，請回答「未找到相關資訊」。
            請務必依據情境分流，條列式說明。

            【法規資料】：{context}
            【櫃員提問】：{question}
            """)

            final_prompt = prompt_template.format_messages(context=context_text, question=prompt)

            # 加入重試機制呼叫 LLM
            try:
                response = llm.invoke(final_prompt)
                final_answer = response.content + f"\n\n**📑 參考法規出處：** {', '.join(source_breadcrumbs)}"
            except Exception as e:
                final_answer = "⚠️ 伺服器目前忙碌中，請稍後再試。"

            # 顯示結果與摺疊面板
            st.markdown(final_answer)
            if "伺服器" not in final_answer:
                with st.expander("📄 查看檢索到的法規原文"):
                    st.markdown(source_display)

                # 存入對話紀錄
                st.session_state.messages.append({
                    "role": "assistant",
                    "content": final_answer,
                    "sources": source_display
                })
            else:
                 st.session_state.messages.append({
                    "role": "assistant",
                    "content": final_answer
                })

In [ ]:
# 1. 安裝所需套件
print("📦 正在安裝套件中，請稍候...")
!pip install -q streamlit langchain langchain-google-genai langchain-community sentence-transformers faiss-cpu
!npm install -g localtunnel
print("✅ 安裝完成！請執行下一個 Cell。")

In [ ]:
# 2. 啟動伺服器與隧道
import urllib

print("👉 【第一步】請先複製這串 IP 密碼：")
!wget -q -O - ipv4.icanhazip.com

print("\n👉 【第二步】點擊下方出現的 `your url is: ...` 連結")
print("👉 【第三步】貼上 IP，按下 Submit！")

!streamlit run app.py & npx localtunnel --port 8501

  Stopping...
  Stopping...
Traceback (most recent call last):
  File "/usr/local/bin/streamlit", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1485, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1406, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1873, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1269, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/streamlit/web/bootstrap.py", line 43, in signal_handler
    server.stop()
  File "/usr/local/lib/python3